# DNAGPT: A Generalized Pre-trained Tool for Versatile DNA Sequence Analysis
**ArXivist-generated reproduction notebook**
Paper: https://arxiv.org/abs/2307.05628 (Zhang et al., 2023)
Generated: 2026-07-24

This notebook (1) verifies the DNAGPT **architecture + token language + combined loss**
on CPU, (2) loads the **official pretrained weights** (Google Drive), and (3) fine-tunes
on the **real DeepGSR** genomic-signal-recognition task. Use a **GPU runtime** for step 3
(Runtime > Change runtime type > T4 GPU).

## 1. Upload the repo zip

In [ ]:
from google.colab import files
up = files.upload()          # pick arxiv_2307_005628.zip
!unzip -oq arxiv_2307_005628.zip
%cd arxiv_2307_005628
import os; print('cwd:', os.getcwd(), '| has src/dnagpt:', os.path.isdir('src/dnagpt'))

## 2. Environment + install

In [ ]:
import sys, torch, subprocess
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
r = subprocess.run(["pip","install","-q","-e",".","gdown"], capture_output=True, text=True)
print((r.stdout or r.stderr)[-400:])

## 3. Verify the architecture (CPU, no weights) — 8 tests

Checks DNAGPT's contributions mechanically: non-overlap k-mer → N/k tokens, the
19,530-kmer vocab (S1.3), dual heads, joint seq+number forward, combined loss
`L = 0.01·MSE + CE`.

In [ ]:
open("conftest.py","a").close()
import subprocess
r = subprocess.run(["python","-m","pytest","tests/","-q"], capture_output=True, text=True)
print(r.stdout[-600:]); print(r.stderr[-200:] if r.returncode else "all tests passed")

## 4. Download official weights (Google Drive)

Drive IDs are baked into `data/download.py`. `dna_gpt0.1b_m` is the base model
(matches paper's 0.1B); `classification` is already fine-tuned on AATAAA PAS.

In [ ]:
!python data/download.py --weights dna_gpt0.1b_m
# optional (already-fine-tuned head): !python data/download.py --weights classification

## 5. Load the pretrained model + confirm weight-compat

The from-scratch architecture is **weight-compatible** with the official checkpoint —
expect `matched~76 unexpected=0`.

In [ ]:
from src.dnagpt.data.tokenizer import DNAGPTTokenizer
from src.dnagpt.models.dnagpt import DNAGPT, DNAGPTConfig
tok = DNAGPTTokenizer(k=6)
cfg = DNAGPTConfig.from_variant("M", vocab_size=19564, seq_len=606, num_classes=2)
model = DNAGPT.from_pretrained("checkpoints/dna_gpt0.1b_m.pth", cfg, device=str(device))
print(model)

## 6. Download the REAL DeepGSR data (Zenodo, ~255 MB)

The authors' exact PAS FASTA. Builds the paper's 6:1.5:2.5 splits — for human
PAS(AATAAA), 11,302 positives + 11,302 hard-negatives (matches Table S5).

In [ ]:
!python data/download.py --gsr human_pas_aataaa

## 7. Fine-tune on real GSR (GPU)

Human PAS(AATAAA). Paper DNAGPT-M target: **91.51% acc / 82.99 MCC** (Table S2).
The `[done]` line is labeled 'real DeepGSR data' when real CSVs are present (and
loudly 'SYNTHETIC' if they are not).

In [ ]:
!python train.py --config configs/config.yaml --task human_pas_aataaa

## 8. Paper results for comparison (Table S2 / S4)

In [ ]:
paper = {
  "DNAGPT-M Human PAS(AATAAA) acc": 0.9151, "DNAGPT-M Human PAS(AATAAA) mcc": 0.8299,
  "DNAGPT-M Human TIS(ATG) acc": 0.9746,
  "DNAGPT-B-512 Human PAS(AATAAA) acc": 0.9320, "DNAGPT-B-512 Human TIS(ATG) acc": 0.9802,
}
for k, v in paper.items(): print(f"  {k:40s} {v}")
print("\nPaste your fine-tuned acc/mcc back to ArXivist for the Stage 6 comparison.")

## What to do next

- Try other tasks: `--task human_tis_atg` (paper 97.46%), `--task mouse_pas_aataaa`, etc.
  (run `python data/download.py --gsr <task>` first).
- Evaluate a checkpoint: `python evaluate.py --config configs/config.yaml --checkpoint checkpoints/best.pt`
- Paste your real acc/mcc back to ArXivist (Stage 6).

**Recipe (Fig S4):** AdamW β(0.9,0.98), lr 3e-5, cosine+warmup, 10 epochs, wd 1e-1, bs 8.